# 00 Pilot: Keyword Search and Hand Check
**Purpose:** Before the full scrape, estimate how often negative reviews of AI-disclosed games mention generative AI, and test a keyword search.

**Data:** 312 English negative reviews (Jan 2024 – Dec 2025) from the 5 lowest-rated AI-disclosed games at the time, scraped Nov 4, 2026.

**Outputs:** `pilot.csv` (development set, excluded from all testing) and `pilot_hits_labeled.csv` (hand-checked keyword hits).

**Key results:** 8 of 15 keyword hits (53%) were about generative AI; estimated rate 2.6% (95% CI 1.3–5.0%).

## Set up

In [1]:
#Imports
import pandas as pd
import numpy as np
import os

## 1. Pull the Pilot

In [2]:
# Pull the pilot (already run on Nov 4, 2026; left commented out so re-running the notebook doesn't change the data)

# %run steam_reviews_scraper.py --appids 2738870 3484300 2687970 2943670 3444080 --max-per-game 100 --out pilot.csv

## 2. Load and Check

In [3]:
# Load the pilot and confirm it matches what was scraped
df = pd.read_csv("pilot.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(df["appid"].value_counts())

assert len(df) == 312, f"Expected 312 pilot reviews, found {len(df)}"
assert df["appid"].nunique() == 5, "Pilot should cover 5 games"
assert (df["voted_up"] == False).all(), "Pilot should contain only negative reviews"
assert df["recommendationid"].is_unique, "Review IDs should be unique"

Loaded: 312 rows, 18 columns
appid
3484300    100
2687970    100
2738870     48
2943670     41
3444080     23
Name: count, dtype: int64


## 3. Keyword Search

In [4]:
# Keyword search for AI mentions
# This pattern is fixed here and reused unchanged as the keyword baseline, so it isn't tuned on test data
AI_PATTERN = r"\bai\b|a\.i\.|ai-generated|generative|genai|slop|midjourney|chatgpt|artificial intelligence"

hits = df[df["review"].str.contains(AI_PATTERN, case=False, regex=True)].copy()
print(f"{len(hits)} of {len(df)} reviews match the keyword pattern ({len(hits)/len(df):.1%})")
print(hits.groupby("appid").size())

15 of 312 reviews match the keyword pattern (4.8%)
appid
2687970     3
2943670     1
3484300    11
dtype: int64


## 4. Export for Hand-Check

In [5]:
# Export keyword hits for hand labeling
# Only writes the file if it doesn't exist, so re-running never erases the hand labels
out_file = "pilot_hits_labeled.csv"
if not os.path.exists(out_file):
    hits["label"] = ""
    hits["note"] = ""
    hits[["recommendationid", "appid", "review", "label", "note"]].to_csv(out_file, index=False)
    print(f"Exported {len(hits)} hits to {out_file}")
else:
    print(f"{out_file} already exists; not overwriting hand labels")

pilot_hits_labeled.csv already exists; not overwriting hand labels


## 5. Hand-Check Results

In [6]:
# Hand check results
# Codes: G = generative AI, E = enemy/NPC/bot behavior, N = keyword matched but no AI meaning, ? = unclear
labeled = pd.read_csv("pilot_hits_labeled.csv")

assert len(labeled) == len(hits), "Every keyword hit should be in the labeled file"
assert labeled["label"].notna().all(), "Some hits are still unlabeled"
assert set(labeled["label"]) <= {"G", "E", "N", "?"}, f"Unexpected labels: {set(labeled['label'])}"

print(labeled["label"].value_counts())

def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval for a proportion (works well for small counts)."""
    p = k / n
    center = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half = z * np.sqrt(p*(1-p)/n + z**2 / (4*n**2)) / (1 + z**2 / n)
    return center - half, center + half

n_g = (labeled["label"] == "G").sum()
precision = n_g / len(labeled)
rate = n_g / len(df)
low, high = wilson_ci(n_g, len(df))
print(f"\nKeyword precision: {n_g} of {len(labeled)} hits ({precision:.0%}) were generative AI")
print(f"Estimated generative-AI rate: {rate:.1%} (95% CI {low:.1%} to {high:.1%})")

label
G    8
E    6
N    1
Name: count, dtype: int64

Keyword precision: 8 of 15 hits (53%) were generative AI
Estimated generative-AI rate: 2.6% (95% CI 1.3% to 5.0%)


## 6. Investigate Edge Cases

In [7]:
# The non-G hits and their notes become examples for the labeling guide
for _, row in labeled[labeled["label"] != "G"].iterrows():
   print(f"[{row['label']}] {row['note'] if pd.notna(row['note']) else ''}\n   {row['review'][:150]}...\n")

[E] 
   This is not a fun game to play in its current state, there is some minor fun to be had with building a house and gear. But the human enemies are all S...

[E] 
   Just a generic survival wanna be game with stupid AI that just runs straight at you
the aiming is over sensitive no matter what settings you use ...fo...

[E] 
   (Update: It's better, but still way too buggy to recommend.)
Guns feels great, but the AI is so broken you'll just get aim botted from bushes before e...

[E] pretty sure it's E but AI mapping? What is that
   Well... let's start with the good, if you have a pretty beefy PC the game is gorgeous(def disable motion blur otherwise its a mess), The voice acting ...

[E] 
   I really wanted to like Dreadzone. It has a gritty, post-apocalyptic vibe concept that could’ve been a hit—but right out of the gate, the pacing felt ...

[E] 
   Let me start off by stating I actually really like the idea of this game a lot. The idea environment is pretty, but the sun rays a

## Decisions
- Complaints about enemy, NPC, or bot behavior ("the AI just runs at you") count as quality complaints, not generative AI complaints.
- Reviews are labeled by the reviewer's stated reason so "AI random gen skins" counts as generative AI whether or not AI was actually used.
- "Slop" counts as generative AI only when AI is named or clearly implied.
- Keyword precision was 53%, so a keyword search alone is not reliable; this motivates the Claude classifier.
- These 312 reviews are the development set as the Claude prompt is tuned only on them, and they are removed from all test data.
- Bakery Cafe Simulator was in the pilot but later dropped from the final game list (only 23 negative reviews in the window).